# Inspect a reference telescope file

Verify the notebook environment and its pairing with a separate reference-data checkout.
We will check the exact input bytes, inspect native observation metadata, and save a small provenance report.
This starter does not perform reduction or establish science-product quality.

First follow the repository README and `datasets/README.md`. Use the environment in `uv.lock`.
Inputs are opened read-only; results go under this tutorial checkout's `_build/` directory.


## Locate the example and data

The default input tree is `../tolteca_ref_data/data_lmt`, beside the tutorials repository.
Set `TOLTECA_DATA_LMT` before starting Jupyter if your data live elsewhere, including a TolTECA deployment's `run/data_lmt`.
The manifest identifies a reference acquisition independently of the Python environment.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import tomllib
from pathlib import Path

from netCDF4 import Dataset

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "datasets/reference.toml").is_file()
)
manifest = tomllib.loads((root / "datasets/reference.toml").read_text())
expected = manifest["telescope_scan"]
data_root = (
    Path(os.environ.get("TOLTECA_DATA_LMT", root.parent / "tolteca_ref_data/data_lmt"))
    .expanduser()
    .resolve()
)
input_path = data_root / expected["path"]
if not input_path.is_file():
    raise FileNotFoundError(
        f"Missing {input_path}. Follow datasets/README.md or set TOLTECA_DATA_LMT."
    )
print(f"Input: {expected['path']}")
print(f"Python: {platform.python_version()}")

## Verify the payload

The expected byte count and SHA-256 come from the Git LFS pointer at the manifest's upstream revision.
A pointer file, missing download, or different acquisition must fail before we interpret it.


In [ ]:
if input_path.stat().st_size != expected["size_bytes"]:
    raise ValueError(
        "Input size differs from the manifest; check the Git LFS download."
    )
with input_path.open("rb") as stream:
    input_hash = hashlib.file_digest(stream, "sha256").hexdigest()
if input_hash != expected["sha256"]:
    raise ValueError("Input checksum differs from the pinned reference acquisition.")
print(f"Verified {expected['size_bytes']:,} bytes; SHA-256 {input_hash}")

## Read observation identity and native array metadata

Read the observation, sub-observation, and scan identifiers directly from this file.
Inspect the shared time-sample count and native time/angle units without changing or reducing the arrays.
These checks describe this pinned input only.


In [ ]:
with Dataset(input_path, mode="r") as dataset:
    identity = {
        "obsnum": int(dataset.variables["Header.Dcs.ObsNum"][...]),
        "subobsnum": int(dataset.variables["Header.Dcs.SubObsNum"][...]),
        "scannum": int(dataset.variables["Header.Dcs.ScanNum"][...]),
    }
    samples = len(dataset.dimensions["time"])
    arrays = {}
    for name, unit in (("TelTime", "sec"), ("TelAzAct", "rad"), ("TelElAct", "rad")):
        variable = dataset.variables[f"Data.TelescopeBackend.{name}"]
        if variable.shape != (expected["time_samples"],) or variable.units != unit:
            raise ValueError(f"Unexpected shape or native units for {name}")
        arrays[name] = {"shape": list(variable.shape), "units": variable.units}

if identity != {name: expected[name] for name in identity}:
    raise ValueError("Observation identity differs from the reference manifest.")
if samples != expected["time_samples"]:
    raise ValueError("Time-sample count differs from the reference manifest.")
print(identity)
print(f"Time samples: {samples}")
print(json.dumps(arrays, indent=2))

## Save the result and provenance

The report records the input identity, data-repository revision, notebook-source hash, environment-lock hash, and library versions.
The output directory belongs to tutorials and is ignored by Git. A successful run establishes this setup and file-inspection workflow; full scientific recipes will add their own calibration and quality checks.


In [ ]:
notebook_path = root / "workflows/inspect-reference-data/inspect_reference_data.ipynb"
summary = {
    "reference_repository": manifest["repository"],
    "reference_revision": manifest["revision"],
    "input_path": expected["path"],
    "input_sha256": input_hash,
    "input_size_bytes": expected["size_bytes"],
    "notebook_sha256": hashlib.sha256(notebook_path.read_bytes()).hexdigest(),
    "environment_lock_sha256": hashlib.sha256(
        (root / "uv.lock").read_bytes()
    ).hexdigest(),
    "python": platform.python_version(),
    "netCDF4": importlib.metadata.version("netCDF4"),
    "identity": identity,
    "time_samples": samples,
    "arrays": arrays,
}
output = root / "_build/inspect-reference-data/summary.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(f"All checks passed. Saved {output.relative_to(root)}")